# Pokémon GO Trainer Stat Model

**Revised.** The first version of this notebook fit a linear regression of trainer level
against three lifetime stats and extrapolated to level 50. That model is wrong in a way
worth documenting, so this notebook now does two things: it reproduces the original
result, then shows exactly why it fails and what to use instead.

The interactive version of everything below lives in `site/` — see the readme.


## 1. Setup and load

`trainingdata.csv` is headerless: `trainername, level, battleswon, distancewalked, pokemoncaught`.
390 trainers captured from a personal friends list. Distance is *assumed* to be km — the
original capture didn't record the unit setting.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

COLS = ['trainername', 'level', 'battleswon', 'distancewalked', 'pokemoncaught']
METRICS = ['battleswon', 'distancewalked', 'pokemoncaught']
CAP = 50  # the game's level ceiling

raw = pd.read_csv('trainingdata.csv', header=None, names=COLS)

print(f'{len(raw)} trainers, levels {raw.level.min()}-{raw.level.max()}')
print(f'{(raw.level == CAP).sum()} of them at the level-{CAP} cap')
display(raw[METRICS].describe().round(1))

## 2. The bug in the original cleaning

The first version removed outliers with an IQR filter applied to four columns **in sequence**,
each pass operating on the survivors of the last. Endgame players hold the largest lifetime
totals, so every pass preferentially removes them. Watch the level-50 population.


In [ ]:
def remove_outliers(df, column):
    q1, q3 = df[column].quantile(0.25), df[column].quantile(0.75)
    iqr = q3 - q1
    return df[(df[column] >= q1 - 1.5 * iqr) & (df[column] <= q3 + 1.5 * iqr)]

cleaned = raw.copy()
print(f"{'start':<16} n={len(cleaned):>3}   at cap: {(cleaned.level == CAP).sum():>2}")
for col in ['level'] + METRICS:
    cleaned = remove_outliers(cleaned, col)
    print(f'{col:<16} n={len(cleaned):>3}   at cap: {(cleaned.level == CAP).sum():>2}')

print(f'\ndropped {len(raw) - len(cleaned)} of {len(raw)} rows ({100 * (len(raw) - len(cleaned)) / len(raw):.1f}%)')
print('LEVEL-50 TRAINERS: 73 -> ' f'{(cleaned.level == CAP).sum()}  <-- the model now extrapolates to a level it never saw')

## 3. Scoring the original predictions against reality

Refit on the cleaned data exactly as the original did, then compare its level-by-level
predictions to the *actual median* of real trainers at each level in the full dataset.


In [ ]:
# The original fit on an 80% train split (random_state=42) — reproduced exactly here,
# so these numbers match the published predictions_level_35_to_50.csv.
from sklearn.model_selection import train_test_split

models = {}
for m in METRICS:
    X_tr, _, y_tr, _ = train_test_split(
        cleaned[['level']], cleaned[m], test_size=0.2, random_state=42)
    models[m] = LinearRegression().fit(X_tr, y_tr)

levels = np.arange(35, CAP + 1)
pred = pd.DataFrame({'level': levels})
for m in METRICS:
    pred[m] = models[m].predict(pred[['level']]).round(2)

# sanity-check against the committed CSV the original notebook produced
published = pd.read_csv('predictions_level_35_to_50.csv')
assert np.allclose(pred.pokemoncaught, published['Pokémon Caught'], atol=0.01), 'drifted from published'
print('reproduces predictions_level_35_to_50.csv exactly')

actual = raw.groupby('level')[METRICS].median()
counts = raw.groupby('level').size().rename('n')

comp = pred.set_index('level').join(actual, rsuffix='_actual').join(counts)
out = comp[['n', 'pokemoncaught', 'pokemoncaught_actual']].round(0).astype(int)
out['off_by'] = (comp.pokemoncaught_actual / comp.pokemoncaught).map(
    lambda r: f'x{r:.1f} low' if r >= 1 else f'x{1/r:.1f} high')
display(out)

cap_ratio = comp.loc[CAP, 'pokemoncaught_actual'] / comp.loc[CAP, 'pokemoncaught']
print(f'At the cap the model is off by {cap_ratio:.1f}x (too low).')

## 4. Growth compounds — a straight line is the wrong shape

Compare three models on the **full** 390-row dataset: the original linear fit, a log-linear
fit (`log(stat) ~ level`, back-transformed), and a nonparametric baseline that just predicts
the median of each level. R² is reported in the original units so all three are comparable.


In [ ]:
rows = []
for m in METRICS:
    X, y = raw[['level']], raw[m]

    lin = LinearRegression().fit(X, y)
    r2_lin = r2_score(y, lin.predict(X))

    pos = raw[raw[m] > 0]
    log = LinearRegression().fit(pos[['level']], np.log(pos[m]))
    r2_log = r2_score(y, np.exp(log.predict(X)))

    r2_med = r2_score(y, raw.level.map(raw.groupby('level')[m].median()))

    rows.append({'metric': m, 'linear': r2_lin, 'log-linear': r2_log,
                 'median-per-level': r2_med, 'growth/level': np.exp(log.coef_[0])})

r2_table = pd.DataFrame(rows).set_index('metric').round(3)
print('R^2 in original units, full 390-row dataset (higher is better):')
display(r2_table)

In [ ]:
# The straight line crosses zero inside the observed range.
for m in METRICS:
    lin = LinearRegression().fit(raw[['level']], raw[m])
    zero_at = -lin.intercept_ / lin.coef_[0]
    print(f'{m:<16} predicts 0 at level {zero_at:.1f}; negative below (real data goes down to {raw.level.min()})')

## 5. Level 50 is a censored cap, not a value

19% of the cohort shares one level. Within it the stats span an order of magnitude or more,
so for those trainers `level` carries no information at all. This is a hard ceiling on how
good *any* level-based model can be — not something better fitting can fix.


In [ ]:
at_cap = raw[raw.level == CAP]
spread = at_cap[METRICS].describe(percentiles=[.1, .25, .5, .75, .9]).round(0)
spread.loc['spread'] = (at_cap[METRICS].max() / at_cap[METRICS].min()).round(0)
display(spread)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m in zip(axes, METRICS):
    ax.hist(at_cap[m], bins=20, color='#41d8c6', edgecolor='#0a0d1c')
    ax.set_title(f'{m} at level {CAP} (n={len(at_cap)})')
    ax.set_xlabel(m)
plt.tight_layout()
plt.show()

## 6. What to use instead: level-independent ratios

Catches per km and battles per catch describe *how* someone plays rather than how far along
they are. They are unaffected by the level cap, which makes them the most useful signal in
this dataset — and they segment into recognisable playstyles.


In [ ]:
ratios = raw[(raw.distancewalked > 0) & (raw.pokemoncaught > 0)].copy()
ratios['catches_per_km'] = ratios.pokemoncaught / ratios.distancewalked
ratios['battles_per_1k_catches'] = 1000 * ratios.battleswon / ratios.pokemoncaught

print(ratios[['catches_per_km', 'battles_per_1k_catches']].describe().round(1))

plt.figure(figsize=(9, 6))
sc = plt.scatter(ratios.catches_per_km, ratios.battles_per_1k_catches,
                 c=ratios.level, cmap='viridis', alpha=.75, edgecolors='none')
plt.colorbar(sc, label='Trainer level')
plt.xlim(0, ratios.catches_per_km.quantile(.98))
plt.ylim(0, ratios.battles_per_1k_catches.quantile(.98))
plt.xlabel('Catches per km walked')
plt.ylabel('Battles won per 1,000 catches')
plt.title('Playstyle — independent of level')
plt.grid(alpha=.2)
plt.show()

## Conclusion

The original conclusion — that level predicts these stats well enough to forecast them —
does not survive checking. Three findings replace it:

1. **The chained IQR filter destroyed the sample it was meant to clean**, taking the level-50
   population from 73 trainers to 1 and leaving the model to extrapolate blind. Its level-50
   prediction is ~4.8× too low.
2. **Growth compounds at roughly ×1.3 per level**, so a straight line is the wrong shape; it
   even predicts negative totals inside the observed range. A log-linear fit is better, and
   simply taking the median of each level is better still on two of three metrics.
3. **The level cap censors a fifth of the data**, putting a hard ceiling on any level-based
   model. Level-independent ratios avoid the problem entirely.

The dataset is small, self-selected from one friends list, and undated. It is a good
descriptive picture of a real player cohort and a poor basis for prediction — which is
exactly what the dashboard in `site/` is built to show.
